Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Data Loading (identik dengan isolation_forest.ipynb)

In [ ]:
# Path relatif dari folder "Anomaly Detection BMKG" ke folder "data"
file_path = '../data/bmkg/gabungan_2008_2025.csv'

# Load data
df = pd.read_csv(file_path)

print(f"Dataset berhasil dimuat dengan dimensi: {df.shape}")
df.head()

Data Preparation (identik dengan isolation_forest.ipynb)

In [ ]:
# Menampilkan jumlah awal
print(f"Jumlah data awal: {len(df)}")

# Batas koordinat geografis Indonesia (secara kasar / bounding box)
lat_min, lat_max = -11.0, 6.0
lon_min, lon_max = 95.0, 141.0

# 1. Filter hanya koordinat yang berada di dalam bounding box Indonesia
df = df[(df['latitude'] >= lat_min) & (df['latitude'] <= lat_max) &
        (df['longitude'] >= lon_min) & (df['longitude'] <= lon_max)]
print(f"Jumlah data setelah difilter wilayah Indonesia: {len(df)}")

# 2. Menghapus semua baris yang memiliki nilai kosong (NaN)
df = df.dropna()
print(f"Jumlah data setelah drop missing values: {len(df)}")

# 3. Merapikan (reset) index setelah ada penghapusan baris
df = df.reset_index(drop=True)

df.info()

Feature Engineering & Preprocessing (identik dengan isolation_forest.ipynb)

In [ ]:
# Pilih fitur sesuai kriteria proyek
features = ['mag', 'depth', 'latitude', 'longitude']
X = df[features]

# Standarisasi menggunakan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# Perbandingan Antar Nilai Contamination
Melatih beberapa model Isolation Forest dengan nilai *contamination* berbeda di atas data yang sama persis (`X_scaled_df`), lalu membandingkan hasilnya.

In [ ]:
# Daftar nilai contamination yang ingin dibandingkan
contaminations = [0.001, 0.005, 0.01, 0.02, 0.05]

results = {}
total_data = len(X_scaled_df)

for cont in contaminations:
    iso_forest = IsolationForest(contamination=cont, random_state=42)
    labels = iso_forest.fit_predict(X_scaled_df)
    scores = iso_forest.decision_function(X_scaled_df)

    total_anomaly = (labels == -1).sum()
    total_normal = (labels == 1).sum()
    persentase = (total_anomaly / total_data) * 100

    results[cont] = {
        'model': iso_forest,
        'labels': labels,
        'scores': scores,
        'total_normal': total_normal,
        'total_anomaly': total_anomaly,
        'persentase_anomaly': persentase,
        'mean_score_normal': scores[labels == 1].mean(),
        'mean_score_anomaly': scores[labels == -1].mean(),
    }

    print(f"[ Contamination {cont} ]")
    print(f"Total Normal   : {total_normal}")
    print(f"Total Anomaly  : {total_anomaly}")
    print(f"Persentase     : {persentase:.3f}%")
    print(f"Mean Score (Normal)  : {results[cont]['mean_score_normal']:.4f}")
    print(f"Mean Score (Anomaly) : {results[cont]['mean_score_anomaly']:.4f}")
    print("-" * 40)

## Tabel Ringkasan Perbandingan

In [ ]:
summary = pd.DataFrame([
    {
        'Contamination': cont,
        'Total Normal': r['total_normal'],
        'Total Anomaly': r['total_anomaly'],
        'Persentase Anomaly (%)': round(r['persentase_anomaly'], 3),
        'Mean Score Normal': round(r['mean_score_normal'], 4),
        'Mean Score Anomaly': round(r['mean_score_anomaly'], 4),
    }
    for cont, r in results.items()
])

display(summary)

## Visualisasi: Jumlah Anomali per Nilai Contamination

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary['Contamination'].astype(str), summary['Total Anomaly'], color='crimson')
plt.title('Jumlah Data Anomali per Nilai Contamination')
plt.xlabel('Contamination')
plt.ylabel('Total Anomaly')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

## Visualisasi: Sebaran Lokasi Anomali per Nilai Contamination

In [ ]:
fig, axes = plt.subplots(1, len(contaminations), figsize=(6 * len(contaminations), 5), sharex=True, sharey=True)

for ax, cont in zip(axes, contaminations):
    labels = results[cont]['labels']
    normal_mask = labels == 1
    anomaly_mask = labels == -1

    ax.scatter(df.loc[normal_mask, 'longitude'], df.loc[normal_mask, 'latitude'],
               c='lightgray', alpha=0.4, s=2, label='Normal')
    ax.scatter(df.loc[anomaly_mask, 'longitude'], df.loc[anomaly_mask, 'latitude'],
               c='red', alpha=0.8, s=10, label='Anomali')
    ax.set_title(f'Contamination = {cont}')
    ax.set_xlabel('Longitude')
    ax.legend()

axes[0].set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Distribusi Anomaly Score per Nilai Contamination

In [ ]:
fig, axes = plt.subplots(1, len(contaminations), figsize=(6 * len(contaminations), 4), sharey=True)

for ax, cont in zip(axes, contaminations):
    scores = results[cont]['scores']
    threshold = pd.Series(scores).quantile(cont)
    ax.hist(scores, bins=80, color='skyblue', edgecolor='black')
    ax.axvline(threshold, color='red', linestyle='dashed', linewidth=2,
               label=f'Threshold ({threshold:.3f})')
    ax.set_title(f'Contamination = {cont}')
    ax.set_xlabel('Anomaly Score')
    ax.legend()

axes[0].set_ylabel('Frekuensi')
plt.tight_layout()
plt.show()